In [ ]:
from google.colab import drive
# Google Drive ???
drive.mount('/content/drive')

In [ ]:
!pip install dtaidistance

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from dtaidistance import dtw

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import ADASYN
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from imblearn.metrics import specificity_score # specificity_score ???
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Conv1D, Flatten
import seaborn as sns

In [ ]:
df_X_train = pd.read_csv('/content/drive/My Drive/PhalangesOutlinesCorrect/X_train_sample2.csv')
df_y_train = pd.read_csv('/content/drive/My Drive/PhalangesOutlinesCorrect/y_train_sample2.csv')
df_X_test = pd.read_csv('/content/drive/MyDrive/PhalangesOutlinesCorrect/X_test_original.csv')
df_y_test = pd.read_csv('/content/drive/MyDrive/PhalangesOutlinesCorrect/y_test_original.csv')

In [ ]:
X_train = df_X_train.values
y_train = df_y_train.values.reshape(-1)  # 1?? ??(??)? ??
X_test = df_X_test.values
y_test = df_y_test.values.reshape(-1)  # 1?? ??(??)? ??

In [ ]:
# ?? ???? ?? ??? ??
target_counts = pd.Series(y_train).value_counts()
minority_class = target_counts.idxmin()
majority_class = target_counts.idxmax()

# ?? ??? ?? ??
X_minority = X_train[y_train == minority_class]
X_major = X_train[y_train == majority_class]

In [ ]:
# ?? ???? ?? ???? ?? ???
#scaler_minority = MinMaxScaler()
#scaler_majority = MinMaxScaler()

#X_minority_norm = scaler_minority.fit_transform(X_minority)
#X_majority_norm = scaler_majority.fit_transform(X_major)

In [ ]:
# Xcenter ?? (?? ??? ??)
Xcenter = np.mean(X_minority, axis=0)

In [ ]:
Xcenter

In [ ]:
wpd_distances = []
for sample in X_minority:
    distance = dtw.distance(Xcenter, sample)
    wpd_distances.append(distance)

wpd_distances = np.array(wpd_distances)
wpd_mean = np.mean(wpd_distances)

safe_samples = X_minority[wpd_distances <= wpd_mean]
noisy_samples = X_minority[wpd_distances > wpd_mean]

In [ ]:
print(len(safe_samples), len(noisy_samples))

In [ ]:
# ??? ?
n_generate = len(X_major)-len(X_minority)
samples_per_safe_point = n_generate // len(safe_samples)
remainder = n_generate % len(safe_samples)

In [ ]:
augmented_samples = []

# 1? ??
for sample in safe_samples:
    for _ in range(samples_per_safe_point):
        alpha = np.random.rand()
        new_sample = alpha * sample + (1 - alpha) * Xcenter
        augmented_samples.append(new_sample)

# ??? 4? ??
for i in range(remainder):
    sample = safe_samples[i % len(safe_samples)]
    alpha = np.random.rand()
    new_sample = alpha * sample + (1 - alpha) * Xcenter
    augmented_samples.append(new_sample)

augmented_samples = np.array(augmented_samples)


In [ ]:
# ?? ???? ?? scaler_minority ???? ????
X_minority_final = np.vstack([X_minority, augmented_samples])
y_minority_final = np.array([minority_class] * X_minority_final.shape[0])

# ?? ???? scaler_minority ???? ???
X_majority_final = X_major
y_majority_final = np.array([majority_class] * X_majority_final.shape[0])
# ?? ??
X_synthetic = np.vstack([X_minority_final, X_majority_final])
y_synthetic = np.concatenate([y_minority_final, y_majority_final])


In [ ]:
def predict_with_logistic_regression(X_train, y_train, X_test):
    model = LogisticRegression()
    model.fit(X_train, y_train)
    return model.predict(X_test)

def predict_with_cart(X_train, y_train, X_test):
    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)
    return model.predict(X_test)

def predict_with_knn(X_train, y_train, X_test, k=3):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    return model.predict(X_test)

def predict_with_xgboost(X_train, y_train, X_test):
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train, y_train)
    return model.predict(X_test)

def predict_with_lstm(X_train, y_train, X_test):
    model = Sequential()
    model.add(LSTM(50, activation='relu', input_shape=(X_train.shape[1], 1)))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    X_train_reshaped = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test_reshaped = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    model.fit(X_train_reshaped, y_train, epochs=50, batch_size=32, verbose=0)
    return (model.predict(X_test_reshaped) > 0.5).astype(int)

def predict_with_cnn(X_train, y_train, X_test):
    model = Sequential()
    model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    X_train_reshaped = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test_reshaped = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    model.fit(X_train_reshaped, y_train, epochs=50, batch_size=32, verbose=0)
    return (model.predict(X_test_reshaped) > 0.5).astype(int)

In [ ]:
# ?? ?? ? ??
predictions = {
    "Logistic Regression": predict_with_logistic_regression(X_synthetic, y_synthetic, X_test),
    "CART": predict_with_cart(X_synthetic, y_synthetic, X_test),
    "KNN": predict_with_knn(X_synthetic, y_synthetic, X_test),
    "XGBoost": predict_with_xgboost(X_synthetic, y_synthetic, X_test),
    "LSTM": predict_with_lstm(X_synthetic, y_synthetic, X_test),
    "CNN": predict_with_cnn(X_synthetic, y_synthetic, X_test),
}
# ?? ?? ??? ?? ???
results = []
for model_name, pred_y in predictions.items():
    accuracy = accuracy_score(y_test, pred_y)
    recall = recall_score(y_test, pred_y)
    f1 = f1_score(y_test, pred_y)
    specificity = specificity_score(y_test, pred_y)
    conf_matrix = confusion_matrix(y_test, pred_y)
    results.append([accuracy, f1, recall, specificity, conf_matrix])
    print(f"{model_name} ?? ??:")
    print(f"???: {accuracy:.4f}, F1: {f1:.4f}, Recall: {recall:.4f}, Specificity: {specificity:.4f}")
    print("Confusion Matrix:")
    print(conf_matrix)

    plt.figure(figsize=(4, 3))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'{model_name} Confusion Matrix')
    plt.show()


In [ ]:
# ?? ??? DataFrame?? ??
results_df = pd.DataFrame(results, columns=["Accuracy", "F1", "Recall", "Specificity", "Confusion Matrix"], index=predictions.keys())
model_results = results_df.T
# ??? ??
print("\n?? ?? ?? ??:")
print(model_results)

# ??? CSV ??? ??
model_results.to_csv("/content/drive/My Drive/PhalangesOutlinesCorrect/results/msmote_models_result.csv")